# Sweden CBA / PBA hotspot analysis — notebook 1 of 3: build dataframes — EXIOBASE 3 v3.8.2

**Project:** RISE pre-study for Region Stockholm. This notebook is the stable national baseline. A later step will layer Stockholm / Rest-of-Sweden disaggregation on top.

**What this notebook does.** A national-level hotspot analysis of Sweden across three dimensions:

1. **Economic scale indicator** — gross output (M EUR) for the production perspective, direct final-demand expenditure (M EUR) for the consumption perspective. These are complementary monetary scale indicators, not full upstream economic footprints equivalent to the Leontief-traced GHG and material accounts.
2. **GHG emissions** — total CO₂-eq (kt), AR5 GWP100; direct territorial flows for PBA and Leontief-traced footprints for CBA/source/destination attribution.
3. **Material footprint** — kt, aggregated into the four Anthesis primary categories; direct domestic extraction for PBA and Leontief-traced footprints for CBA/source/destination attribution.

For GHG and material, we compute full production-based (PBA) and consumption-based (CBA) footprint accounts, identify top sectors, and attribute Sweden's CBA to source countries and sectors. For the economic dimension, we compute complementary monetary scale indicators: Swedish gross output on the production side and direct final-demand expenditure on the consumption side. We trace Sweden's territorial GHG and material impacts to final consumers via Leontief tracing, while treating the economic destination as direct market sales.

A **tier-1 upstream linkage analysis** is included as sections 11 (attributable to Swedish final demand) and 12 (production-oriented foreign input dependency). Tier-1 identifies the direct first-tier foreign purchases made by Swedish sectors, using the Z transaction matrix without Leontief tracing. This is a diagnostic complement to the full Leontief-traced accounts; it does not replace them.

**Database:** EXIOBASE 3 v3.8.2 (`IOT_2022_pxp.zip`) — Creative Commons Attribution Share Alike 4.0 International

## Key structural notes (v3.8 vs v3.10)

EXIOBASE v3.8.2 (and earlier) organises environmental data differently from v3.9 onwards:

- Only two extensions are exposed: `exio.satellite` (1113 rows, uncharacterised stressors + factor inputs) and `exio.impacts` (126 rows, characterised indicators).
- The satellite extension carries only **22 GHG rows** (vs 420 in v3.10.1) and **does not distinguish fossil from biogenic CO2**.
- Consequently, this notebook reports a single **Total GHG** column (AR5 GWP100) rather than a fossil / biogenic split. A by-gas breakdown is printed once as a diagnostic.
- Material extraction: 217 "Domestic Extraction Used" rows in satellite, mapped to the four Anthesis primary categories (biomass, fossil, metals, minerals).
- The `Value Added` row in the impacts extension is loaded as a reference check only. It is not used as the primary economic indicator.

**Outputs** are written to `./outputs/` (CSV tables).


## Primer — concepts used in this notebook

This section gives plain-English definitions of the technical terms that appear throughout the notebook. It is aimed at readers who have not worked with multi-regional input-output (MRIO) analysis before. Readers comfortable with EXIOBASE can skip it.

### What is an input-output (IO) model?

An IO model is an accounting framework that describes how industries in an economy buy from and sell to each other. If Swedish steel mills need electricity, cement and ore to produce steel, the IO model records:

- How many euros of electricity, cement and ore the steel sector buys per euro of steel produced (**technical coefficients**).
- How much of the steel's output goes to other industries (cars, construction) versus ultimately to **final demand** (households, government, investment, exports).

A **multi-regional** IO model extends this to several countries at once, so that supply chains crossing borders can be traced. Swedish steel used in German cars, which are then bought by Swedish households, appears in the data as a chain of flows.

### What is EXIOBASE?

EXIOBASE 3 is one of several widely used global MRIO databases, alongside Eora and WIOD. The v3.8.2 release used here covers:

- **49 regions** — 44 countries plus 5 "rest of world" aggregates.
- **200 product sectors** per region.
- A full set of **environmental extensions**: material extraction, greenhouse gas emissions, water use, land use, employment, and many others.

The total system has 49 × 200 = 9,800 (region, sector) pairs. Most matrices in this notebook are 9,800 × 9,800.

### Economic indicators used in this notebook

This notebook uses two complementary monetary scale indicators. They are **not** full upstream economic footprints and are **not** directly comparable with the Leontief-traced GHG and material accounts.

**Gross output (M EUR)** is the total monetary value of what a sector produces, including intermediate inputs purchased from other sectors. It is used for the **production-side economic indicator**. Gross output is a measure of the monetary scale of each sector's production activity.

**Direct expenditure (M EUR)** is Sweden's total spending on each consumed product category, summed across all direct source countries. It is used for the **consumption-side economic indicator**. Direct expenditure captures the monetary value of Sweden's final demand for each product category and direct source country. It avoids double-counting intermediate flows, but it does **not** trace where the upstream economic activity embodied in the product occurred. For example, if Sweden imports a German car, the transaction appears as a payment to Germany's motor vehicle sector. The economic activity in Poland, China, or elsewhere that contributed to assembling that car is not separately attributed in this indicator.

**Why not value added?** Value added reflects income distribution — the share of output retained as wages, profits, and taxes. Labour-intensive service sectors generate high value added per euro of output and therefore always dominate value added rankings, even when their gross production value is modest relative to goods-producing sectors. In a CE context this gives a misleading picture of physical economic scale.

**On aggregate double-counting.** Gross output cannot be summed across all sectors to produce a single national total without double-counting (intermediate goods appear at each production stage). Sector-level rankings in this notebook are valid. The net trade summary (section 10) therefore covers only GHG and material dimensions.

### PBA vs CBA — the two perspectives

**Production-based account (PBA)** attributes an impact to the sector that physically causes it inside Swedish territory. Swedish steelmaking emits GHG from blast furnaces in Sweden: those emissions count as Swedish PBA, regardless of where the steel is ultimately used. The production-side economic indicator is gross output of Swedish sectors.

**Consumption-based account (CBA)** attributes the same impact to the final consumer. If Swedish-made steel ends up in a phone bought by a Swedish household, the emissions count as Swedish CBA. If that steel is exported to Germany, the emissions count as German CBA. The consumption-side economic indicator is Sweden's direct expenditure on each consumed product category.

For GHG and material, CBA uses the full Leontief inverse to trace impacts through all upstream supply-chain tiers. For the economic dimension, no Leontief tracing is applied. These are therefore not fully parallel accounting systems, and should not be interpreted as such.

### The mathematical objects behind the calculations

- **Z** — the inter-industry transaction matrix. `Z[i, j]` is the monetary flow from sector i to sector j as an intermediate input.
- **A** — technical coefficients: `A = Z · diag(x)⁻¹`.
- **Y** — final demand matrix: rows are (producing region, sector), columns are (consuming region, final-demand category).
- **L** — the Leontief inverse `L = (I − A)⁻¹`. `L[i, j]` is the total output of sector i required per euro of final demand delivered by sector j, including all indirect supply-chain requirements.
- **F** — direct environmental flow of each sector (kt GHG or kt material). Used for physical dimensions only.
- **S** — direct intensity `S = F · diag(x)⁻¹`. Used for physical dimensions only.
- **M** — total multiplier `M = S · L`. Used for physical dimensions only.

With these objects:

$$D^{PBA}_{GHG,\,mat} = F \text{ restricted to producing region}$$

$$D^{CBA}_{GHG,\,mat} = M \cdot \text{diag}(y^{SE}) \quad \text{(full upstream tracing)}$$

$$D^{prod\_econ} = x^{SE} \quad \text{(Swedish gross output by sector)}$$

$$D^{cons\_econ} = y^{SE} \text{ grouped by consumed sector} \quad \text{(direct expenditure)}$$


## 1. Setup

In [ ]:
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import pymrio

from IPython.display import display

warnings.filterwarnings("ignore")


In [ ]:
# ---------- Path configuration -------------------------------------------
EXIOBASE_PATH = r"C:\EXIOBASE3\IOT_2022_pxp.zip"

# ---------- Focus region and analysis knobs -------------------------------
REGION = "SE"          # ISO code for Sweden in EXIOBASE
TOP_N  = 15            # how many entries to show in top-N tables / charts
TOP_C  = 5             # how many source / destination countries to show per sector

OUTPUT_DIR = Path("./outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# ---------- Unit conventions (reference) ---------------------------------
# EXIOBASE v3.8.2 raw units:
#   economic core       : M.EUR  (current prices)
#   satellite GHG rows  : kg (CO2, CH4, N2O, SF6 physical; HFC, PFC already kg CO2-eq)
#   satellite DEU rows  : kt
#   impacts "Value Added": M.EUR  (loaded as reference only; not used as primary indicator)
#
# Reporting units used in this notebook:
#   economic (PBA) : M.EUR, gross output of Swedish producing sectors
#   economic (CBA) : M.EUR, direct final-demand expenditure by consumed product category
#   GHG            : kt CO2-eq  (kg -> kt: divide by 1e6)
#   material       : kt

print(f"EXIOBASE path  : {EXIOBASE_PATH}")
print(f"Focus region   : {REGION}")
print(f"Top-N          : {TOP_N}")
print(f"Top-C          : {TOP_C}")
print(f"Output folder  : {OUTPUT_DIR.resolve()}")


## 2. Load EXIOBASE

We parse the archive with `pymrio.parse_exiobase3`. In v3.8.2 the pre-computed extension multipliers (`S`, `M`) and footprint accounts (`D_cba`, `D_pba`) are already stored inside the archive, which saves one expensive computation for us.

In [ ]:
def load_exio(path):
    print(f"Parsing EXIOBASE from: {path}")
    t0 = time.time()
    exio = pymrio.parse_exiobase3(path=path)
    print(f"  parsed in {time.time()-t0:.1f}s")
    print(f"  A shape : {None if exio.A is None else exio.A.shape}")
    print(f"  Y shape : {exio.Y.shape}")
    print(f"  x shape : {None if exio.x is None else exio.x.shape}")
    regions = exio.get_regions().tolist()
    sectors = exio.get_sectors().tolist()
    print(f"  regions : {len(regions)} ({regions[:5]}...)")
    print(f"  sectors : {len(sectors)}")
    for ext_name in ['satellite', 'impacts']:
        ext = getattr(exio, ext_name, None)
        if ext is not None and ext.F is not None:
            S_ok = ext.S is not None
            M_ok = ext.M is not None
            print(f"  ext '{ext_name}': F={ext.F.shape}, "
                  f"S={'Y' if S_ok else 'N'}, M={'Y' if M_ok else 'N'}")
        else:
            print(f"  ext '{ext_name}': NOT FOUND")
    return exio

exio = load_exio(EXIOBASE_PATH)


### What just loaded — a quick tour

The printout above shows:

- `A shape: (9800, 9800)` — the technical coefficients matrix. 9,800 rows and columns because 49 regions × 200 products.
- `Y shape: (9800, 343)` — final demand. 343 columns = 49 regions × 7 final-demand categories per region.
- `x shape: (9800, 1)` — total output per (region, sector).
- `regions: 49` — the country/aggregate list. Sweden (`SE`) is one of them.
- `sectors: 200` — the product list that each region has.
- `ext 'satellite': F=(1113, 9800), S=Y, M=Y` — 1,113 stressor rows (emissions, extractions, factor inputs) with direct coefficients (S) and multipliers (M) already present.
- `ext 'impacts': F=(126, 9800), S=Y, M=Y` — 126 characterised indicators (aggregated impacts, including the pre-computed `Value Added` and `GHG emissions AR5 (GWP100)` rows we use for cross-checks).

The `S=Y, M=Y` flags are important. They tell us the database ships with pre-computed direct intensities and total multipliers, so we do not need to recompute them. We only need to rebuild the Leontief inverse `L` (which is too large to ship inside the archive at ~770 MB).


## 3. Compute the Leontief inverse L

`calc_system()` computes `A = Z · diag(x)⁻¹` (if not already present) and the Leontief inverse `L = (I − A)⁻¹`. This is the single memory-heavy step (~3 GB peak on the 9800×9800 pxp system). We call it only once per kernel session.

In [ ]:
if getattr(exio, "L", None) is None:
    print("Computing Leontief inverse L (can take a few minutes)...")
    t0 = time.time()
    exio.calc_system()
    print(f"  L shape: {exio.L.shape}   ({time.time()-t0:.1f}s)")
else:
    print(f"L already computed: {exio.L.shape}. Skipping.")


### Why we compute L

The Leontief inverse $L = (I - A)^{-1}$ is the single most important object in MRIO analysis. It converts final demand (what Sweden buys) into total production (what the world has to produce, directly and indirectly, to satisfy that demand).

A simple intuition: if you buy a cup of coffee for 1 EUR, that cup includes not just the coffee beans but also:

- Electricity used to roast the beans.
- Steel in the roasting machine.
- Diesel used in the ship that brought the beans.
- Paper in the cup.
- Chemicals in the cup's lining.
- Plus the supply chains of each of those inputs.

The Leontief inverse captures all of these tiers of indirect requirements in one matrix. Any row `L[i, j]` tells us the total output of sector i that must occur, anywhere in the world, for 1 Euro of sector j's output to be delivered to final demand.

Computing L takes about 25 seconds for the 9,800 × 9,800 system and uses a few gigabytes of RAM. We do this once per kernel session. Everything downstream is cheap matrix multiplication against L.


## 4. Indicator helpers

### 4.1 Region slicing

In [ ]:
EXIO_RC_NAMES = ["region", "sector"]
EXIO_FD_NAMES = ["region", "category"]

def _coerce_index(idx, names=None):
    """Return a pandas Index / MultiIndex with predictable level names.

    EXIOBASE objects sometimes come back as a MultiIndex and sometimes as a plain
    Index of tuples. The latter looks identical when printed, but arithmetic
    alignment can silently fail. This helper normalises both cases.
    """
    if isinstance(idx, pd.MultiIndex):
        out = idx
    else:
        vals = list(idx)
        if vals and isinstance(vals[0], tuple):
            out = pd.MultiIndex.from_tuples(vals)
        else:
            out = pd.Index(idx)
            if names and len(names) == 1:
                out.name = names[0]
            return out

    if names is not None and len(names) == out.nlevels:
        out = out.set_names(names)
    return out

def normalize_series(s, index_names=None):
    out = s.copy()
    if index_names is not None:
        out.index = _coerce_index(out.index, index_names)
    return out

def normalize_frame(df, index_names=None, column_names=None):
    out = df.copy()
    if index_names is not None:
        out.index = _coerce_index(out.index, index_names)
    if column_names is not None:
        out.columns = _coerce_index(out.columns, column_names)
    return out

def cols_of(df, region):
    """Select columns of a (region, *)-columned DataFrame for one region."""
    df_n = normalize_frame(df, column_names=EXIO_FD_NAMES)
    return df_n.loc[:, pd.IndexSlice[region, :]]

def safe_pct(two_col_array):
    """Row-wise percentages with protection against zero totals."""
    arr = np.asarray(two_col_array, dtype=float)
    denom = arr.sum(axis=1, keepdims=True)
    return np.divide(arr, denom, out=np.zeros_like(arr), where=denom != 0) * 100

# Sanity check:
Y_tmp = normalize_frame(exio.Y, index_names=EXIO_RC_NAMES, column_names=EXIO_FD_NAMES)
_se_sectors = cols_of(Y_tmp, REGION).columns.get_level_values("category").tolist()
print(f"Sweden has {len(_se_sectors)} final-demand categories. First 5:")
for s in _se_sectors[:5]:
    print(f"  {s}")


#### Why we normalise the index labels

pymrio sometimes returns a pandas object whose row labels are a `MultiIndex` with explicit level names (`region`, `sector`), and sometimes returns the same labels as a flat `Index` of tuples. Printed to the screen they look identical, but pandas arithmetic treats them differently. Multiplying a flat-tuple Series by a MultiIndex Series produces silent NaN alignment failures that cascade through the downstream attribution and ruin the country × sector breakdowns.

The `_coerce_index`, `normalize_series`, and `normalize_frame` helpers force every key object (L, Y, all F/M/S matrices) into a true MultiIndex with the correct level names before any matrix multiplication happens. This was the root cause of the Section 8–11 failures in the previous run.

`safe_pct` is a small convenience to avoid dividing by zero when a group has no flow.


### 4.2 GHG aggregation (total, AR5 GWP100)

We aggregate the 22 GHG rows in the satellite extension into a single **Total GHG** stream in kt CO2-eq, using AR5 GWP100 factors. Unit handling per EXIOBASE v3.8.2:

- CO2, CH4, N2O, SF6 rows are physical kg → multiplied by AR5 GWP factor.
- HFC, PFC rows are already **kg CO2-eq** → factor = 1.

As a diagnostic, we also provide a per-gas breakdown (CO2 / CH4 / N2O / F-gases) so the composition can be inspected once.

In [ ]:
GWP_AR5 = {"CO2": 1, "CH4": 28, "N2O": 265, "SF6": 23500, "HFC": 1, "PFC": 1}

# All 22 GHG-candidate rows present in v3.8.2 satellite (confirmed via diagnostic).
GHG_ROWS_V382 = {
    "CO2": [
        "CO2 - combustion - air",
        "CO2 - non combustion - Cement production - air",
        "CO2 - non combustion - Lime production - air",
        "CO2 - agriculture - peat decay - air",
    ],
    "CH4": [
        "CH4 - combustion - air",
        "CH4 - non combustion - Extraction/production of (natural) gas - air",
        "CH4 - non combustion - Extraction/production of crude oil - air",
        "CH4 - non combustion - Mining of antracite - air",
        "CH4 - non combustion - Mining of bituminous coal - air",
        "CH4 - non combustion - Mining of coking coal - air",
        "CH4 - non combustion - Mining of lignite (brown coal) - air",
        "CH4 - non combustion - Mining of sub-bituminous coal - air",
        "CH4 - non combustion - Oil refinery - air",
        "CH4 - agriculture - air",
        "CH4 - waste - air",
    ],
    "N2O": [
        "N2O - combustion - air",
        "N2O - agriculture - air",
    ],
    "SF6": ["SF6 - air"],
    "HFC": ["HFC - air"],
    "PFC": ["PFC - air"],
}

def aggregate_ghg(flow_matrix):
    """Aggregate all 22 GHG rows to total kt CO2-eq (kg input, divided by 1e6)."""
    present = set(flow_matrix.index.tolist())
    out = None
    missing = []
    for gas, rows in GHG_ROWS_V382.items():
        gwp = GWP_AR5[gas]
        for row in rows:
            if row in present:
                contrib = flow_matrix.loc[row] * gwp
                out = contrib if out is None else out + contrib
            else:
                missing.append(row)
    if missing:
        print(f"  [aggregate_ghg] missing rows: {missing}")
    return out / 1e6   # kg -> kt

def aggregate_ghg_by_gas(flow_matrix):
    """Return a DataFrame with rows [CO2, CH4, N2O, F-gases] in kt CO2-eq."""
    gas_to_group = {"CO2": "CO2", "CH4": "CH4", "N2O": "N2O",
                    "SF6": "F-gases", "HFC": "F-gases", "PFC": "F-gases"}
    present = set(flow_matrix.index.tolist())
    accum = {g: None for g in ["CO2", "CH4", "N2O", "F-gases"]}
    for gas, rows in GHG_ROWS_V382.items():
        gwp = GWP_AR5[gas]
        target = gas_to_group[gas]
        for row in rows:
            if row in present:
                contrib = flow_matrix.loc[row] * gwp
                accum[target] = contrib if accum[target] is None else accum[target] + contrib
    out = pd.DataFrame({k: (v / 1e6 if v is not None else 0.0)
                        for k, v in accum.items()}).T
    return out


### 4.3 Material aggregation — Anthesis 4 categories

v3.8.2's satellite has 217 "Domestic Extraction Used" (DEU) rows, all in kt. We map them to the four Anthesis primary categories by the group name (second field of the row name):

| Anthesis category | Satellite group name(s) |
|---|---|
| biomass  | Primary Crops, Forestry, Grazing, Fodder crops, Crop residues, Fishery |
| fossil   | Fossil Fuel: Total |
| metals   | Metal Ores |
| minerals | Non-Metallic Minerals |

In [ ]:
ANTHESIS_GROUP_MAPPING = {
    "biomass":  ["Primary Crops", "Forestry", "Grazing", "Fodder crops",
                 "Crop residues", "Fishery"],
    "fossil":   ["Fossil Fuel: Total"],
    "metals":   ["Metal Ores"],
    "minerals": ["Non-Metallic Minerals"],
}

DEU_PREFIX = "Domestic Extraction Used - "

def build_material_mapping(satellite_F, verbose=True):
    """Map Used-DEU rows in satellite to Anthesis categories by group name."""
    rows = satellite_F.index.tolist()
    deu_rows = [r for r in rows if r.startswith(DEU_PREFIX)]
    mapping = {cat: [] for cat in ANTHESIS_GROUP_MAPPING}
    unmapped = []
    for r in deu_rows:
        remainder = r[len(DEU_PREFIX):]
        group = remainder.split(" - ")[0] if " - " in remainder else remainder
        placed = False
        for cat, groups in ANTHESIS_GROUP_MAPPING.items():
            if group in groups:
                mapping[cat].append(r)
                placed = True
                break
        if not placed:
            unmapped.append((group, r))
    if verbose:
        total = 0
        for cat, rws in mapping.items():
            print(f"  {cat:<9}  {len(rws):>4} rows")
            total += len(rws)
        print(f"  total mapped: {total} / {len(deu_rows)} DEU rows")
        if unmapped:
            unique_groups = sorted(set(g for g, _ in unmapped))
            print(f"  UNMAPPED groups: {unique_groups}")
    return mapping

MATERIAL_MAP = build_material_mapping(exio.satellite.F)

def aggregate_materials(flow_matrix, mapping=MATERIAL_MAP):
    """Aggregate material flow matrix into (4 categories x cols)."""
    out = pd.DataFrame(0.0, index=list(mapping.keys()), columns=flow_matrix.columns)
    for cat, rows in mapping.items():
        rows_present = [r for r in rows if r in flow_matrix.index]
        if rows_present:
            out.loc[cat] = flow_matrix.loc[rows_present].sum(axis=0)
    return out


### 4.4 Economic indicators — gross output and direct expenditure

Two complementary monetary scale indicators are used in this notebook instead of value added.

**Gross output (M EUR)** is the total monetary value of what a sector produces, including intermediate inputs purchased from other sectors. It is used as the **production-side monetary scale indicator** because it captures the full monetary scale of a sector's production activity. It is useful for identifying sectors where large volumes of production value may be relevant for circularity interventions. However, it should be interpreted as a sector-scale proxy, not as a direct estimate of value loss or avoided economic activity. Gross output is not a full upstream economic footprint.

**Direct expenditure (M EUR)** is Sweden's total spending on each consumed product category, summed across all direct source countries. It is the **consumption-side monetary scale indicator**. It captures what Sweden pays its direct suppliers for each product category. It avoids double-counting intermediate flows but does not trace where upstream economic activity embodied in those products occurred.

Both indicators use EXIOBASE without multiplying by a satellite intensity coefficient.

- **Production side:** `exio.x.xs("SE", level="region")` — Swedish sector gross output vector.
- **Consumption side:** `y_SE.groupby(level="sector").sum()` — Sweden's final demand summed by consumed product category.

**On aggregate double-counting.** Gross output cannot be summed across all sectors to produce a single national total without double-counting (intermediate goods appear at each production stage). Sector-level rankings in this notebook are valid. The net trade summary (section 10) covers only GHG and material dimensions, where PBA and CBA are directly comparable physical accounts.


In [ ]:
VA_ROW = "Value Added"
# Value Added is retained as a reference metric only.
# Primary economic indicators are gross output (PBA) and direct expenditure (CBA).
assert VA_ROW in exio.impacts.F.index, f"Row '{VA_ROW}' missing from impacts extension"


#### Note on value added

Value added is **not** used as the primary economic indicator in this notebook. The rationale is that value added reflects income distribution (labour and capital shares), so labour-intensive service sectors systematically dominate VA rankings even when their gross production value is modest. In a CE prioritisation context, gross output is often more useful than value added because it better reflects the monetary scale of production activity affected by material flows, product turnover, and circularity interventions. It should still be interpreted as a scale proxy, not as a direct estimate of value leakage.

Gross output and direct expenditure are used instead, as described in section 4.4.


## 5. Build sector-level indicator vectors

For the **physical dimensions (GHG and material)** we build three objects:

- **F** (direct flows) — impact generated by each producing sector per its actual output. Used for PBA.
- **S** (direct intensities) — impact per euro of output. Used to attribute source countries in the CBA.
- **M** (total multipliers) — total impact per euro of final demand, including all upstream tiers via the Leontief inverse. Used for CBA via `D_cba = M × diag(y)`.

The **economic dimension** is handled separately and does not use F, S, or M. It uses gross output (`x`) directly for the production side and Sweden's final demand vector (`y`) for the consumption side.


In [ ]:
sat = exio.satellite
imp = exio.impacts

# Normalise the key economic tables once so later matrix operations keep a
# consistent (region, sector) / (region, category) labelling throughout.
L_df = normalize_frame(exio.L, index_names=EXIO_RC_NAMES, column_names=EXIO_RC_NAMES)
Y_df = normalize_frame(exio.Y, index_names=EXIO_RC_NAMES, column_names=EXIO_FD_NAMES)

# ---------- Direct impacts (F) ----------
F_ghg = normalize_series(aggregate_ghg(sat.F), EXIO_RC_NAMES)            # Series, kt CO2e
F_mat = normalize_frame(aggregate_materials(sat.F), None, EXIO_RC_NAMES) # DataFrame (4, 9800), kt

# ---------- Multipliers (M = S @ L) — GHG and material only ----------
M_ghg = normalize_series(aggregate_ghg(sat.M), EXIO_RC_NAMES)            # Series, kt CO2e / M EUR
M_mat = normalize_frame(aggregate_materials(sat.M), None, EXIO_RC_NAMES) # DataFrame (4, 9800)

# ---------- Direct intensities (S) — GHG and material only ----------
S_ghg = normalize_series(aggregate_ghg(sat.S), EXIO_RC_NAMES)            # Series, kt CO2e / M EUR
S_mat = normalize_frame(aggregate_materials(sat.S), None, EXIO_RC_NAMES) # DataFrame (4, 9800)

print("--- Global direct totals ---")
print(f"  Total GHG : {F_ghg.sum()/1000:>10,.0f} Mt CO2e")
print(f"  Material  : {F_mat.sum().sum()/1000:>10,.0f} Mt")


### 5.1 Cross-check — manual GHG aggregation vs pre-computed AR5 GWP100

The impacts extension carries `GHG emissions AR5 (GWP100) | GWP100 (IPCC, 2010)`. If our manual aggregation is correctly handling units and GWP factors, the two should match closely. Any discrepancy > 1% should be investigated.

In [ ]:
GHG_AR5_ROW = "GHG emissions AR5 (GWP100) | GWP100 (IPCC, 2010)"
if GHG_AR5_ROW in imp.F.index:
    pre = imp.F.loc[GHG_AR5_ROW] / 1e6    # kg -> kt CO2e
    manual = F_ghg                        # kt CO2e
    diff_pct = (manual.sum() - pre.sum()) / pre.sum() * 100
    print(f"Manual total GHG      : {manual.sum():>12,.0f} kt CO2e")
    print(f"Pre-computed AR5 total: {pre.sum():>12,.0f} kt CO2e")
    print(f"Difference            : {diff_pct:+6.2f}%")
    if abs(diff_pct) > 1.0:
        print("  WARNING: difference > 1%. Check GHG row list / units.")
    else:
        print("  OK: manual aggregation is consistent with the pre-computed AR5 GWP100.")
else:
    print(f"NOTE: '{GHG_AR5_ROW}' not found in impacts. Skipping cross-check.")


### 5.2 Diagnostic — GHG composition by gas (Sweden)

A one-off breakdown of Sweden's PBA GHG into CO2 / CH4 / N2O / F-gases, so we can sanity-check the composition against known inventory shares (CO2 typically ~80% of Sweden's anthropogenic GHG).

In [ ]:
by_gas_F = aggregate_ghg_by_gas(sat.F)                             # DataFrame (4, 9800) kt CO2e
by_gas_SE_PBA = by_gas_F.xs(REGION, level="region", axis=1).sum(axis=1)
total_SE = by_gas_SE_PBA.sum()
print("Sweden PBA GHG composition by gas:")
for gas, val in by_gas_SE_PBA.items():
    print(f"  {gas:<8}: {val:>10,.0f} kt CO2e  ({val/total_SE*100:5.1f}%)")
print(f"  {'TOTAL':<8}: {total_SE:>10,.0f} kt CO2e")


#### Interpreting the by-gas composition

Sweden's PBA composition in EXIOBASE v3.8.2 shows approximately 65% CO2, 23% CH4, 8% N2O, and 4% F-gases. This differs from what the Swedish national inventory typically reports under UNFCCC (where CO2 is about 80% of total GHG).

The by-gas composition is internally consistent with the EXIOBASE v3.8.2 GHG rows and confirmed by the cross-check against the pre-computed AR5 GWP100 indicator (within 0.04%). However, it should **not** be directly compared with Sweden's national inventory without further investigation, for two reasons:

1. **EXIOBASE v3.8.2 does not separate fossil and biogenic CO2.** The CO2 rows aggregate combustion-related CO2 regardless of whether the fuel is fossil or biogenic. This makes it impossible to determine from v3.8.2 alone whether the CO2 share difference from the national inventory reflects biogenic emissions, sectoral boundary differences, or both.

2. **EXIOBASE uses its own sectoral and territorial structure**, which does not map one-to-one onto UNFCCC reporting boundaries. Differences in how transport, agriculture, and waste sectors are defined can shift gas-share figures materially.

The GHG results in this notebook should therefore be interpreted as EXIOBASE total GHG on the selected AR5 GWP100 basis, not as a fossil-only or national-inventory-equivalent total.


## 6. PBA hotspot analysis — Sweden

In [ ]:
# Swedish gross output per sector (PBA economic indicator)
x_SE = (exio.x
        .xs(REGION, level="region")
        .squeeze()
        .rename("Gross output (M EUR)"))
# Reindex to match the sector order used by F_ghg and F_mat
F_ghg_SE       = F_ghg.xs(REGION, level="region")
F_mat_SE       = F_mat.xs(REGION, level="region", axis=1)
F_mat_total_SE = F_mat_SE.sum(axis=0)
x_SE = x_SE.reindex(F_ghg_SE.index).fillna(0.0)

pba_SE = pd.DataFrame({
    "Gross output (M EUR)":  x_SE,
    "GHG total (kt CO2e)":   F_ghg_SE,
    "Biomass (kt)":          F_mat_SE.loc["biomass"],
    "Fossil (kt)":           F_mat_SE.loc["fossil"],
    "Metals (kt)":           F_mat_SE.loc["metals"],
    "Minerals (kt)":         F_mat_SE.loc["minerals"],
    "Material total (kt)":   F_mat_total_SE,
})
pba_SE.index.name = "Product sector"

print("=== Sweden production-side indicators ===")
print(f"  Sum of sector gross output, not GDP-like total : {x_SE.sum()/1000:>10,.1f} B EUR")
print(f"  Total territorial GHG                          : {F_ghg_SE.sum():>10,.0f} kt CO2e")
print(f"  Total domestic material extraction             : {F_mat_total_SE.sum():>10,.0f} kt")

pba_SE.to_csv(OUTPUT_DIR / "pba_sweden_by_sector.csv")


### How to read the hotspot tables — sector naming and codes

The EXIOBASE product sector list uses mostly-descriptive names, often ending in a two-digit number in parentheses. Those numbers are **NACE Rev. 1.1 / ISIC division codes** — the European industrial classification system.

#### Codes that appear in the Sweden hotspot results

**Service sectors:**

| Code | Sector name | What it covers |
|---|---|---|
| **45** | Construction | All building and civil engineering work. Extremely material-intensive per euro of output. |
| **55** | Hotels and restaurants | Accommodation, food and beverage service activities. |
| **64** | Post and telecommunications | Postal, courier, and telecommunications services. |
| **70** | Real estate services | Buying, selling, rental, and management of property. Includes **imputed rent on owner-occupied dwellings** — a large slice of household housing expenditure in national accounts. |
| **72** | Computer and related services | IT consulting, software development, data processing, hosting. |
| **74** | Other business services | Legal, accounting, management consulting, architecture, engineering, advertising, cleaning, security, and business support services. |
| **80** | Education services | Primary, secondary, higher, adult, and other education, including public-sector education. |
| **85** | Health and social work services | Hospitals, medical and dental practices, nursing, residential care, and social work. Largely public-sector in Sweden. |
| **91** | Membership organisation services | Trade unions, religious organisations, political parties, and professional associations. |
| **92** | Recreational, cultural and sporting services | Film, radio and TV, libraries, museums, sports, and amusement. |

**Industry and manufacturing (high material and GHG per euro):**

| Code | Sector name | What it covers |
|---|---|---|
| **29** | Machinery and equipment n.e.c. | General-purpose machinery, agricultural machinery, machine tools, domestic appliances. |
| **34** | Motor vehicles, trailers and semi-trailers | Cars, trucks, trailers, parts, and accessories. |
| **35** | Other transport equipment | Ships, rail locomotives, aircraft, spacecraft, motorcycles, and bicycles. |

Several sectors appear **without** a parenthetical code because EXIOBASE uses a more granular product-level split than two-digit NACE. Examples:

- **Basic iron and steel and of ferro-alloys and first products thereof** — upstream steel.
- **Cement, lime and plaster** — a single EXIOBASE product combining these three materials.
- **Paper and paper products** — paper manufacturing.
- **Steam and hot water supply services** — district heating and industrial steam, partly biofuel-fired in Sweden.
- **Food waste for treatment: landfill** / **Paper for treatment: landfill** — waste-treatment sectors. Methane emissions from landfill are attributed to these sectors (see note below).
- **Other land transportation services** — trucking, bus, and taxi including most freight road haulage.
- **Sea and coastal water transportation services** — shipping. Sweden has a significant shipping register.
- **Air transport services (62)** — aviation. Swedish-registered airlines and aviation services.

**Primary sectors (high material extraction):**

| Sector | What it covers |
|---|---|
| Iron ores | Domestic iron ore mining (LKAB, northern Sweden). |
| Copper ores and concentrates | Swedish copper mining (Aitik, Boliden). |
| Lead, zinc and tin ores and concentrates | Swedish base metal mining. |
| Stone / Sand and clay | Aggregate mining for construction. |
| Products of forestry, logging and related services | Roundwood and forestry services — a major Swedish primary sector. |
| Cattle / Raw milk / Poultry | Livestock farming. |

**Why landfill sectors appear high in GHG PBA.** EXIOBASE attributes landfill methane to the waste-treatment sector operating the landfill. The sector's output (treatment service) has low monetary value, but the methane (CH4, GWP-28) from decomposing food and paper waste is large in CO2-equivalent terms. This is correct accounting. In the CBA account, those emissions are re-attributed back to the upstream consumption that generated the waste (food products, paper products, household consumption), so the landfill sectors are absent from the CBA top list.


In [ ]:
def show_top(df, col, n=TOP_N):
    return df.nlargest(n, col)

print(f"=== Top {TOP_N} Swedish PBA sectors by GROSS OUTPUT ===")
display(show_top(pba_SE, "Gross output (M EUR)"))


In [ ]:
print(f"=== Top {TOP_N} Swedish PBA sectors by TOTAL GHG ===")
display(show_top(pba_SE, "GHG total (kt CO2e)"))


In [ ]:
print(f"=== Top {TOP_N} Swedish PBA sectors by MATERIAL TOTAL ===")
display(show_top(pba_SE, "Material total (kt)"))


### What the PBA tables tell us about Sweden's territorial profile

Three structural features stand out.

**Gross output reflects the monetary scale of each sector's production activity.** Sectors with large material and energy throughput — manufacturing, construction, wholesale and retail trade — appear prominently. This gives a picture of economic scale that is more relevant to CE intervention than labour-income measures.

**Territorial GHG emissions are concentrated in a handful of industrial and waste sectors.** Steam and hot water supply, two landfill sectors, sea transport, basic iron and steel, agriculture-linked sectors (raw milk, cattle) and cement and lime together cover well over half of Sweden's territorial GHG.

**Material extraction is dominated by mining, forestry, and construction aggregates.** Iron ores, stone, forestry, copper, and sand and clay alone account for over 100 Mt of Sweden's 181 Mt direct extraction. The Northern Sweden mining complex (iron ore, copper, base metals) shows up clearly.

The three dimensions have limited overlap. The top gross-output sector is not the top GHG sector, nor the top material sector. This is the core insight of a three-dimension hotspot analysis: the most important sector depends entirely on which question is asked.

Note that the gross output indicator here is a **monetary scale measure**, not a full upstream economic footprint. Sectors high on this list have large production value; they are not necessarily the sectors with the largest embodied upstream economic activity triggered by Swedish consumption.


## 7. Consumption-side hotspot analysis — Sweden

This section ranks consumed product categories from two perspectives: a monetary scale indicator (direct expenditure) and the full Leontief-traced physical footprints (GHG and material).

**Economic dimension — direct expenditure (monetary scale indicator):**

$$D^{cons}_{econ}[s] = \sum_{(c,\,s)} y^{SE}[(c,\,s)]$$

Sweden's total direct spending on product category s, summed across all source countries. No Leontief tracing is applied. This is a monetary scale indicator, not a full upstream economic footprint.

**GHG and material dimensions — full Leontief-traced CBA footprint:**

$$D^{CBA}_{GHG,\,mat}[(c_{orig}, s)] = M[(c_{orig}, s)] \times y^{SE}[(c_{orig}, s)]$$

where `M = S @ L` is the total multiplier matrix. Aggregating across origin countries gives the CBA footprint per consumed product category. This traces impacts through all supply-chain tiers.


In [ ]:
# Sweden's final demand, summed across final-demand categories
Y_SE = cols_of(Y_df, REGION)                                    # (9800, 7) by (origin, product)
y_SE = normalize_series(Y_SE.sum(axis=1), EXIO_RC_NAMES)        # (9800,) M EUR by (origin, product)

# CBA direct expenditure per consumed product category
# (no Leontief tracing — avoids double-counting intermediate stages)
cba_econ_by_sec = y_SE.groupby(level="sector").sum()            # (200,) M EUR

# CBA GHG and material — still use Leontief multipliers (full supply-chain attribution)
D_cba_ghg = M_ghg * y_SE                                        # (9800,)
D_cba_mat = M_mat.multiply(y_SE, axis=1)                        # (4, 9800)

cba_ghg_by_sec       = D_cba_ghg.groupby(level="sector").sum()
cba_mat_by_sec       = D_cba_mat.T.groupby(level="sector").sum().T   # (4, 200)
cba_mat_total_by_sec = cba_mat_by_sec.sum(axis=0)

cba_SE = pd.DataFrame({
    "Direct expenditure (M EUR)": cba_econ_by_sec,
    "GHG total (kt CO2e)":        cba_ghg_by_sec,
    "Biomass (kt)":               cba_mat_by_sec.loc["biomass"],
    "Fossil (kt)":                cba_mat_by_sec.loc["fossil"],
    "Metals (kt)":                cba_mat_by_sec.loc["metals"],
    "Minerals (kt)":              cba_mat_by_sec.loc["minerals"],
    "Material total (kt)":        cba_mat_total_by_sec,
})
cba_SE.index.name = "Product sector consumed"

print("=== Sweden CBA totals ===")
print(f"  Direct expenditure   : {cba_econ_by_sec.sum()/1000:>10,.1f} B EUR")
print(f"  Total GHG            : {D_cba_ghg.sum():>10,.0f} kt CO2e")
print(f"  Material total       : {D_cba_mat.sum().sum():>10,.0f} kt")

cba_SE.to_csv(OUTPUT_DIR / "cba_sweden_by_sector.csv")


In [ ]:
print(f"=== Top {TOP_N} CBA product sectors in Swedish consumption — DIRECT EXPENDITURE ===")
display(show_top(cba_SE, "Direct expenditure (M EUR)"))


In [ ]:
print(f"=== Top {TOP_N} CBA product sectors in Swedish consumption — TOTAL GHG ===")
display(show_top(cba_SE, "GHG total (kt CO2e)"))


In [ ]:
print(f"=== Top {TOP_N} CBA product sectors in Swedish consumption — MATERIAL TOTAL ===")
display(show_top(cba_SE, "Material total (kt)"))


### Reading the consumption-side hotspot tables

The tables rank consumed product categories by direct expenditure (monetary scale), Leontief-traced GHG footprint, and Leontief-traced material footprint. These are three separate questions, answered with different methods.

**Direct expenditure** shows where Swedish households, government, and investors actually spend money, recorded at the direct source country and product category. It does not trace upstream economic activity beyond the first tier. Housing-related and personal service categories (real estate services, health and social work, construction, wholesale and retail trade) absorb the largest shares of Swedish final demand because those products are purchased in large quantities from domestic suppliers.

**The GHG and material footprints** shift toward goods-intensive categories. Machinery, food, chemicals, and transport equipment carry large upstream GHG and material footprints per euro of final demand. These products are frequently imported, so their upstream impacts land abroad.

**The domestic content of Sweden's CBA (printed in section 8)** shows what fraction of each physical dimension is generated inside Sweden. The domestic content of Sweden’s CBA is printed in section 8. For GHG, the domestic share is typically much lower than the direct expenditure domestic share, because much of Sweden’s consumption footprint occurs abroad. For materials, the domestic share is higher than for GHG but still reflects both domestic extraction and imported material content.


## 8. Source attribution for Sweden's consumption-side account

For each consumption hotspot we want to know where the physical impact is generated and who Sweden pays directly. These two questions have different answers and use different methods.

**Economic dimension — direct market source.** The source is Sweden's direct expenditure vector `y_SE`. No upstream tracing is applied. Each `(country, sector)` entry records how much Sweden pays this supplier directly. This does not reveal where the upstream economic activity embodied in the purchased product occurred.

**GHG and material dimensions — full supply-chain source.** The source is computed via:

$$\text{source}[k,\,(c_{src}, s_{src})] = S[k,\,(c_{src}, s_{src})] \cdot x^{SE}[(c_{src}, s_{src})]$$

where

$$x^{SE} = L \cdot y^{SE}$$

is the total global output at every node in the supply chain driven by Sweden's final demand. This traces impacts through all indirect tiers worldwide.


In [ ]:
# ---------- Total source attribution (aggregate across all of Sweden's consumption) ----------
t0 = time.time()

# For GHG and material: trace through full supply chain (Leontief)
y_for_L = y_SE.reindex(L_df.columns).fillna(0.0)
x_SE_total = pd.Series(
    L_df.to_numpy() @ y_for_L.to_numpy(),
    index=L_df.index,
    name="Output driven by Sweden final demand"
)
print(f"x_SE_total computed in {time.time()-t0:.1f}s. Shape: {x_SE_total.shape}")

src_ghg = pd.Series(
    S_ghg.reindex(L_df.index).fillna(0.0).to_numpy() * x_SE_total.to_numpy(),
    index=L_df.index,
    name="GHG total (kt CO2e)"
)
src_mat = pd.DataFrame(
    S_mat.reindex(columns=L_df.index, fill_value=0.0).to_numpy() * x_SE_total.to_numpy(),
    index=S_mat.index,
    columns=L_df.index
)
src_mat_total = src_mat.sum(axis=0)

# For economic: direct expenditure = y_SE itself (no Leontief tracing)
src_econ = y_SE.reindex(L_df.index).fillna(0.0)
src_econ.name = "Direct expenditure (M EUR)"

# Sanity checks
print(f"\n--- Sanity check (totals must match) ---")
print(f"  Direct expenditure: CBA={cba_econ_by_sec.sum():,.0f}, src sum={src_econ.sum():,.0f}")
print(f"  Total GHG   : CBA={D_cba_ghg.sum():,.0f}, source sum={src_ghg.sum():,.0f}")
print(f"  Material    : CBA={D_cba_mat.sum().sum():,.0f}, source sum={src_mat.sum().sum():,.0f}")


In [ ]:
# ---------- Aggregate by source country ----------
def src_country(s):
    out = s.groupby(level="region").sum().sort_values(ascending=False)
    return out.fillna(0.0)

src_econ_country = src_country(src_econ)
src_ghg_country  = src_country(src_ghg)
src_mat_country  = src_country(src_mat_total)

# Combined country table
src_country_all = pd.DataFrame({
    "Direct expenditure (M EUR)": src_econ_country,
    "GHG total (kt CO2e)":        src_ghg_country,
    "Material total (kt)":        src_mat_country,
}).fillna(0.0)
src_country_all.index.name = "Source country"
src_country_all = src_country_all.sort_values("Direct expenditure (M EUR)", ascending=False)
src_country_all.to_csv(OUTPUT_DIR / "cba_source_by_country.csv")

print(f"=== Top {TOP_N} source countries for Sweden's CBA ===")
display(src_country_all.head(TOP_N))

print("\n--- Sweden's own share (domestic content of CBA) ---")
for col in src_country_all.columns:
    tot = src_country_all[col].sum()
    se  = src_country_all.loc[REGION, col] if REGION in src_country_all.index else 0
    share = (se / tot * 100) if tot != 0 else float("nan")
    print(f"  {col:<32}: {share:5.1f}%")


In [ ]:
# ---------- Top (country, sector) source pairs ----------
def pairs_table(series, n=TOP_N):
    return series.dropna().sort_values(ascending=False).head(n).to_frame("value")

print(f"=== Top {TOP_N} (origin country, origin sector) pairs — Direct expenditure ===")
display(pairs_table(src_econ))


In [ ]:
print(f"=== Top {TOP_N} (origin country, origin sector) pairs — Total GHG ===")
display(pairs_table(src_ghg))


In [ ]:
print(f"=== Top {TOP_N} (origin country, origin sector) pairs — Material total ===")
display(pairs_table(src_mat_total))


### Understanding source attribution

The source tables answer different questions for different dimensions.

For the **economic dimension**, the source is who Sweden pays directly — the `(country, sector)` pairs in Sweden's final demand vector `y_SE`. No supply-chain tracing is applied. The Swedish purchase of a German car appears as a single `(DE, Motor vehicles)` entry. This captures the monetary value of Sweden's final demand for that product and direct supplier. It does **not** reveal where the upstream economic activity embodied in the car occurred; Polish assembly, Chinese components, and Norwegian aluminium are not separately attributed in this indicator.

For **GHG and material**, every unit of impact is attributed to its producing node in the global supply chain via the Leontief inverse. The `CN — Electricity by coal` row, for example, means that China's coal power sector generates that amount of GHG to supply electricity to sectors whose products eventually reach Swedish final demand — even if Sweden never imports Chinese electricity directly. This is the methodologically correct treatment for physical footprint accounting.

The intentional asymmetry between dimensions reflects their different double-counting properties: monetary flows are counted once only if traced at the direct transaction level, while physical impacts must be traced through all tiers to avoid attributing the same emission to both intermediate and final producers.


### 8.2 Per-sector source breakdown

The global (country, sector) pairs above aggregate across all of Sweden's consumption. Here we anchor the breakdown to the same top-N sectors identified in section 7 — in the same ranking order — and show the top-C source countries for each, with their percentage contribution.

This ensures that any sector flagged as a hotspot in section 7 carries its full source-country picture directly alongside it, with no need to cross-reference separate tables.


In [ ]:
sectors_list = exio.get_sectors().tolist()
prod_index   = L_df.columns
print(f"Building per-consumption-sector Y matrix ({len(sectors_list)} columns)...")

# Direct expenditure matrix: for each consumed sector s, which (country, sector) pairs
# does Sweden pay directly? This uses y_SE restricted to rows where sector == s.
Y_SE_by_consumed = pd.DataFrame(0.0, index=prod_index, columns=sectors_list)
y_SE_aligned = y_SE.reindex(prod_index).fillna(0.0)
for s in sectors_list:
    mask = prod_index.get_level_values("sector") == s
    Y_SE_by_consumed.loc[mask, s] = y_SE_aligned.loc[mask].to_numpy()

# Economic attribution: direct expenditure per (origin, sector) per consumed product
attrib_econ = Y_SE_by_consumed   # shape (9800, 200) — no Leontief, no intensity weighting

# GHG and material attribution: Leontief-traced (full supply chain)
t0 = time.time()
X_SE_by_consumed = pd.DataFrame(
    L_df.to_numpy() @ Y_SE_by_consumed.to_numpy(),
    index=L_df.index,
    columns=sectors_list
)
print(f"  X_SE_by_consumed computed in {time.time()-t0:.1f}s. Shape: {X_SE_by_consumed.shape}")

S_mat_total      = S_mat.sum(axis=0).reindex(L_df.index).fillna(0.0)
attrib_ghg       = X_SE_by_consumed.mul(S_ghg.reindex(L_df.index).fillna(0.0), axis=0)
attrib_mat_total = X_SE_by_consumed.mul(S_mat_total, axis=0)

print(f"attrib_econ      : {attrib_econ.shape}  (direct expenditure)")
print(f"attrib_ghg       : {attrib_ghg.shape}")
print(f"attrib_mat_total : {attrib_mat_total.shape}")

print("\n--- Sanity check by consumed product ---")
print(f"  Direct expenditure: {attrib_econ.to_numpy().sum():,.0f}")
print(f"  Total GHG   : {attrib_ghg.to_numpy().sum():,.0f}")
print(f"  Material    : {attrib_mat_total.to_numpy().sum():,.0f}")


### 8.3 Per-sector source breakdown — tables

For each of the top-N consumed sectors identified in section 7, the table shows the total impact for that dimension and the top-C source countries by percentage contribution. Sectors appear in the same rank order as section 7, so there is no need to cross-reference. Both N and C are configured in the setup cell and apply consistently throughout all sections.

In [ ]:
def source_breakdown_table(attrib_matrix, sector_ranking, top_c=TOP_C, unit=""):
    """For each sector in sector_ranking show total impact and top-C source country shares.

    Parameters
    ----------
    attrib_matrix : DataFrame, shape (9800, 200)
        rows = (origin_region, origin_sector), cols = consumed product sector.
        For the economic dimension this is Y_SE_by_consumed (direct expenditure);
        for GHG and material it is the Leontief-traced attrib_ matrix.
    sector_ranking : list of sector names in desired display order.
    top_c : int, number of source countries to include.
    unit : str, label appended to the total column header.
    """
    rows = []
    for sec in sector_ranking:
        if sec not in attrib_matrix.columns:
            continue
        country_totals = attrib_matrix[sec].groupby(level="region").sum()
        total = country_totals.sum()
        if total <= 0:
            continue
        top_countries = country_totals.nlargest(top_c)
        row = {"Consumed sector": sec, f"Total ({unit})": round(total, 1)}
        for rank, (country, val) in enumerate(top_countries.items(), 1):
            row[f"#{rank} source"] = f"{country} ({val / total * 100:.0f}%)"
        rows.append(row)
    return pd.DataFrame(rows).set_index("Consumed sector")

# Sector rankings anchored to section 7 (same order the reader already saw)
top_cba_econ = cba_SE.nlargest(TOP_N, "Direct expenditure (M EUR)").index.tolist()
top_cba_ghg  = cba_SE.nlargest(TOP_N, "GHG total (kt CO2e)").index.tolist()
top_cba_mat  = cba_SE.nlargest(TOP_N, "Material total (kt)").index.tolist()

print(f"=== Top {TOP_N} CBA sectors by DIRECT EXPENDITURE — top {TOP_C} source countries ===")
display(source_breakdown_table(attrib_econ, top_cba_econ, unit="M EUR"))

print(f"\n=== Top {TOP_N} CBA sectors by GHG — top {TOP_C} source countries ===")
display(source_breakdown_table(attrib_ghg, top_cba_ghg, unit="kt CO2e"))

print(f"\n=== Top {TOP_N} CBA sectors by MATERIAL — top {TOP_C} source countries ===")
display(source_breakdown_table(attrib_mat_total, top_cba_mat, unit="kt"))


## 9. Destination attribution for Sweden's production-side account

Where does Sweden's territorial production ultimately serve? This section decomposes Swedish sectors' impacts by the country and final-demand category of their output.

**GHG and material dimensions** use Leontief tracing: Swedish territorial impacts are attributed to the final consumer of Sweden's output, following all downstream supply-chain steps:

$$F^{SE}_{GHG,\,mat}[i, (c_{cons}, cat)] = S^{SE}[i] \times \big(L^{SE\text{-rows}} \cdot Y\big)[i, (c_{cons}, cat)]$$

**Economic dimension** uses direct sales: each Swedish sector's gross output is allocated to its **immediate buyers** using the `Z` and `Y` matrices — not traced to ultimate final consumers. `F_{econ}[k, c]` is the monetary value sold directly from Swedish sector k to country c (as intermediate inputs to other firms or as final demand). This is a direct-market destination indicator, not an ultimate-consumer footprint.

The two methods are not equivalent and should not be compared directly.


In [ ]:
t0 = time.time()
L_SE_rows = L_df.xs(REGION, level="region", axis=0)                         # (200, 9800)
x_SE_dest = pd.DataFrame(
    L_SE_rows.to_numpy() @ Y_df.to_numpy(),
    index=L_SE_rows.index,
    columns=Y_df.columns
)
print(f"x_SE_dest computed in {time.time()-t0:.1f}s. Shape: {x_SE_dest.shape}")

# Swedish direct intensity vectors (GHG and material — still use Leontief destination tracing)
S_ghg_SE = S_ghg.xs(REGION, level="region").reindex(L_SE_rows.index).fillna(0.0)
S_mat_SE = S_mat.xs(REGION, level="region", axis=1).reindex(columns=L_SE_rows.index, fill_value=0.0)

F_ghg_dest       = x_SE_dest.mul(S_ghg_SE, axis=0)              # (200, 343)
F_mat_dest_total = x_SE_dest.mul(S_mat_SE.sum(axis=0), axis=0)  # (200, 343)

# Economic destination: direct sales from Swedish sectors (no Leontief)
# Z[SE, :] recovered from A * diag(x_all); combined with Y[SE, :] for final demand
Y_SE_rows = Y_df.xs(REGION, level="region", axis=0)   # (200, 343) — used in cell 57 too
if getattr(exio, "Z", None) is not None:
    _Z_norm = normalize_frame(exio.Z, EXIO_RC_NAMES, EXIO_RC_NAMES)
    _Z_SE   = _Z_norm.xs(REGION, level="region", axis=0)
else:
    _A_norm = normalize_frame(exio.A, EXIO_RC_NAMES, EXIO_RC_NAMES)
    _x_norm = normalize_series(exio.x.squeeze(), EXIO_RC_NAMES)
    _Z_SE   = _A_norm.xs(REGION, level="region", axis=0).multiply(_x_norm, axis=1)

_Z_to_r = _Z_SE.T.groupby(level="region").sum().T      # (200, 49)
_Y_to_r = Y_SE_rows.T.groupby(level="region").sum().T  # (200, 49)
F_econ_dest = _Z_to_r.add(_Y_to_r, fill_value=0)       # (200, 49) — columns are region codes

# Sanity checks vs PBA totals
print(f"\n--- Sanity check (must match Sweden PBA totals) ---")
print(f"  Gross output   : PBA={x_SE.sum():,.0f}, dest sum={F_econ_dest.to_numpy().sum():,.0f}")
print(f"  Total GHG      : PBA={F_ghg_SE.sum():,.0f}, dest sum={F_ghg_dest.to_numpy().sum():,.0f}")
print(f"  Material total : PBA={F_mat_total_SE.sum():,.0f}, dest sum={F_mat_dest_total.to_numpy().sum():,.0f}")


### Understanding destination attribution

**For GHG and material**, the destination is the final consumer of Sweden's territorial output, traced through all downstream supply-chain steps via the Leontief inverse. A Swedish steel mill's territorial emissions are attributed to, for example, German car buyers if the steel ends up in German cars bought by households. This is a full footprint attribution.

**For the economic dimension**, the destination is Sweden's immediate buyers, derived from the `Z` (intermediate sales) and `Y` (final demand) matrices. `F_econ_dest[k, c]` is the total monetary value that Swedish sector k sells directly to country c — including sales to foreign firms that will further process Sweden's output. This is not equivalent to tracing to ultimate final consumers, and will differ from the GHG and material destination results for the same reason that direct expenditure differs from a full upstream footprint.

The domestic-versus-export share printed in section 9.1 reflects these different methods: for GHG and material, "domestic" means the territorial impact ultimately serves Swedish final demand after Leontief tracing; for the economic dimension, "domestic" means Swedish sectors and Swedish final demand are the immediate buyers.


In [ ]:
# Aggregate destinations to country level
def dest_country(mat):
    return (mat.T.groupby(level="region").sum()
              .T.sum(axis=0)
              .sort_values(ascending=False)
              .fillna(0.0))

ghg_dest_country  = dest_country(F_ghg_dest)
mat_dest_country  = dest_country(F_mat_dest_total)
econ_dest_country = F_econ_dest.sum(axis=0).sort_values(ascending=False)   # already by region

dest_country_all = pd.DataFrame({
    "Gross output (M EUR)": econ_dest_country,
    "GHG total (kt CO2e)":  ghg_dest_country,
    "Material total (kt)":  mat_dest_country,
}).fillna(0.0)
dest_country_all.index.name = "Consumer country"
dest_country_all = dest_country_all.sort_values("Gross output (M EUR)", ascending=False)
dest_country_all.to_csv(OUTPUT_DIR / "pba_destination_by_country.csv")

print(f"=== Top {TOP_N} destination countries for Sweden's PBA ===")
display(dest_country_all.head(TOP_N))

print("\n--- Share of Sweden's PBA retained domestically vs exported ---")
for col in dest_country_all.columns:
    tot = dest_country_all[col].sum()
    se  = dest_country_all.loc[REGION, col] if REGION in dest_country_all.index else 0
    stay = (se / tot * 100) if tot != 0 else float("nan")
    exp  = ((tot - se) / tot * 100) if tot != 0 else float("nan")
    print(f"  {col:<25}: {stay:5.1f}% stays in Sweden, {exp:5.1f}% exported")


In [ ]:
# Aggregate destinations by final-demand category (GHG and material: Leontief-traced)
def dest_by_category(mat):
    return (mat.T.groupby(level="category").sum()
              .T.sum(axis=0)
              .sort_values(ascending=False)
              .fillna(0.0))

# Economic: final-demand categories only (intermediate sales have no fd_category)
econ_dest_by_cat = (Y_SE_rows.T.groupby(level="category").sum().T
                              .sum(axis=0)
                              .sort_values(ascending=False)
                              .fillna(0.0))

dest_cat_all = pd.DataFrame({
    "Direct final-demand sales (M EUR)": econ_dest_by_cat,
    "GHG total (kt CO2e)":                dest_by_category(F_ghg_dest),
    "Material total (kt)":                dest_by_category(F_mat_dest_total),
}).fillna(0.0)
dest_cat_all.index.name = "Final-demand category"
print("=== Sweden's PBA by final-demand CATEGORY ===")
print("(Economic column covers final demand only; intermediate sales excluded.)")
display(dest_cat_all)


### 9.2 Per-sector destination breakdown — tables

For each of the top-N Swedish producing sectors identified in section 6, the table shows where that sector's impact ultimately flows, broken down by the top-C destination countries. Sectors appear in the same rank order as section 6.

In [ ]:
def dest_breakdown_table(F_dest_mat, sector_ranking, top_c=TOP_C, unit="", by_country=False):
    """For each Swedish sector show total impact and top-C destination country shares.

    Parameters
    ----------
    F_dest_mat   : (200, n_dest) DataFrame.
                   If by_country=False, columns are (region, fd_category) MultiIndex
                   and we groupby region. If by_country=True, columns are already
                   region codes (as with F_econ_dest).
    sector_ranking : list of Swedish sector names in desired display order.
    by_country   : set True for F_econ_dest which already has region columns.
    """
    rows = []
    for sec in sector_ranking:
        if sec not in F_dest_mat.index:
            continue
        row_data = F_dest_mat.loc[sec]
        if by_country:
            country_totals = row_data.sort_values(ascending=False)
        else:
            country_totals = row_data.groupby(level="region").sum().sort_values(ascending=False)
        total = country_totals.sum()
        if total <= 0:
            continue
        top_countries = country_totals.nlargest(top_c)
        row = {"Swedish sector": sec, f"Total ({unit})": round(total, 1)}
        for rank, (country, val) in enumerate(top_countries.items(), 1):
            row[f"#{rank} destination"] = f"{country} ({val / total * 100:.0f}%)"
        rows.append(row)
    return pd.DataFrame(rows).set_index("Swedish sector")

# Sector rankings anchored to section 6 (same order the reader already saw)
top_pba_econ = pba_SE.nlargest(TOP_N, "Gross output (M EUR)").index.tolist()
top_pba_ghg  = pba_SE.nlargest(TOP_N, "GHG total (kt CO2e)").index.tolist()
top_pba_mat  = pba_SE.nlargest(TOP_N, "Material total (kt)").index.tolist()

print(f"=== Top {TOP_N} PBA sectors by GROSS OUTPUT — top {TOP_C} destination countries ===")
display(dest_breakdown_table(F_econ_dest, top_pba_econ, unit="M EUR", by_country=True))

print(f"\n=== Top {TOP_N} PBA sectors by GHG — top {TOP_C} destination countries ===")
display(dest_breakdown_table(F_ghg_dest, top_pba_ghg, unit="kt CO2e"))

print(f"\n=== Top {TOP_N} PBA sectors by MATERIAL — top {TOP_C} destination countries ===")
display(dest_breakdown_table(F_mat_dest_total, top_pba_mat, unit="kt"))


## 10. Net trade picture — PBA vs CBA

In [ ]:
net = pd.DataFrame({
    "PBA": [F_ghg_SE.sum(),  F_mat_total_SE.sum()],
    "CBA": [D_cba_ghg.sum(), D_cba_mat.sum().sum()],
}, index=["GHG total (kt CO2e)", "Material total (kt)"])
net["Net trade (PBA - CBA)"] = net["PBA"] - net["CBA"]
net["CBA / PBA ratio"]       = net["CBA"] / net["PBA"]

print("=== Sweden — PBA vs CBA (GHG and material only) ===")
display(net)

print("\nInterpretation:")
print("  CBA > PBA  (ratio > 1) -> Sweden is a NET IMPORTER of this impact")
print("  CBA < PBA  (ratio < 1) -> Sweden is a NET EXPORTER of this impact")
print("\nNote: economic dimension excluded because gross output (PBA) and direct")
print("expenditure (CBA) measure different supply-chain stages and are not")
print("directly comparable as a net-trade pair.")

net.to_csv(OUTPUT_DIR / "sweden_pba_vs_cba.csv")


### Interpreting the net trade picture

The table compares Sweden's production-based and consumption-based accounts for GHG emissions and material extraction. The economic dimension is intentionally excluded: gross output (PBA) and direct expenditure (CBA) measure different points in the supply chain and do not form a valid net-trade pair.

**GHG: CBA / PBA > 1.** Sweden is a net importer of greenhouse gas emissions. The CBA total substantially exceeds the PBA total. The gap represents the offshore GHG footprint — emissions that occur abroad to serve Swedish consumption. See the printed table above for the current figures. The magnitude is broadly in line with Swedish consumption-based GHG estimates reported by Swedish environmental authorities and research organisations, though exact comparability depends on database version, system boundary, base year, and GWP assumptions.

**Material: CBA / PBA > 1.** Sweden is a modest net importer of materials. Despite significant domestic extraction (iron ore, copper, forestry), the country consumes more material than it extracts. The gap is the trade-embodied material deficit. See the printed table above for the current figures.

**Policy implication.** A Swedish climate strategy focused only on territorial emissions misses the offshore GHG footprint. A material strategy focused only on domestic extraction misses the imported material content of consumer goods. For Region Stockholm — where almost no primary extraction occurs and heavy industry is minimal — the CBA perspective is almost always more policy-relevant than PBA.


## 11. Cross-validation against SCB Figur 1 (2022)

This section compares EXIOBASE-derived GHG accounts with the Swedish official
statistics published by SCB in *Utveckling av statistik om växthusgasutsläpp i
industrins värdekedjor* (MI171 2025:5, Figur 1).

**What is compared.**

| EXIOBASE quantity | SCB Figur 1 column | Scope |
|---|---|---|
| `F_ghg_SE` aggregated by sector | Produktionsperspektiv | Scope 1 |
| `cba_ghg_by_sec` aggregated by sector | Konsumtionsperspektiv | Scope 1+2+3 upstream |

**Sectors covered.** The 14 manufacturing sectors (SNI C-codes) in SCB Figur 1.
All values are in Mton CO2e (note: EXIOBASE values are in kt CO2e; divided by 1 000 here).

**Tolerance.** A deviation within ±15% per sector is acceptable. The production
perspective is the tightest test since both sources derive from national environmental
accounts. The consumption perspective will deviate more because EXIOBASE uses global IO
tables while SCB uses Swedish national IO tables.

**Insertion point.** `F_ghg_SE` and `cba_ghg_by_sec` are both defined by this point
(sections 6 and 7 respectively).

In [ ]:
import pandas as pd

# ── 1. SCB reference data (Figur 1, 2022, Mton CO2e) ────────────────────────
scb_ref = pd.DataFrame({
    "sni": [
        "C24", "C10-12", "C19", "C20-21", "C29",
        "C25", "C17-18", "C28", "C16", "C23",
        "C27", "C33", "C22", "C31-32",
    ],
    "label": [
        "Stal- och metallverk", "Livsmedel",
        "Raffinerade petroleumprodukter", "Kemiska produkter och lakemedel",
        "Motorfordon", "Metallvaror utom maskiner",
        "Massa-, papperstillverkning, grafisk reproduktion", "Ovrig maskinindustri",
        "Tra och varor av tra", "Icke-metalliska mineraliska produkter",
        "Industri for elapparatur", "Reparationsverkstader",
        "Gummi- och plastvaruindustri", "Mobler och annan tillverkning",
    ],
    "scb_pba": [4.600772, 0.716082, 2.402616, 1.196590, 0.090498,
                0.221866, 0.767373, 0.090552, 0.389467, 1.910531,
                0.022203, 0.137800, 0.063722, 0.038138],
    "scb_cba": [8.668625, 6.316467, 6.404434, 5.534262, 4.842365,
                2.008448, 3.338970, 3.708805, 1.431778, 0.705386,
                1.467746, 0.110542, 0.899243, 0.927106],
}).set_index("sni")

# ── 2. EXIOBASE sector -> SNI concordance (substring, case-insensitive) ──────
EXIO_TO_SNI = {
    "manufacture of basic metals":                    "C24",
    "manufacture of food products":                   "C10-12",
    "manufacture of beverages":                       "C10-12",
    "manufacture of tobacco":                         "C10-12",
    "manufacture of coke and refined petroleum":      "C19",
    "manufacture of chemicals and chemical products": "C20-21",
    "manufacture of basic pharmaceutical":            "C20-21",
    "manufacture of motor vehicles":                  "C29",
    "manufacture of fabricated metal products":       "C25",
    "manufacture of paper and paper products":        "C17-18",
    "printing and reproduction":                      "C17-18",
    "manufacture of machinery and equipment n.e.c.":  "C28",
    "manufacture of wood and of products of wood":    "C16",
    "manufacture of other non-metallic mineral":      "C23",
    "manufacture of electrical equipment":            "C27",
    "repair and installation":                        "C33",
    "manufacture of rubber and plastic":              "C22",
    "manufacture of furniture":                       "C31-32",
    "other manufacturing":                            "C31-32",
}

def map_sector(name):
    n = name.lower()
    for pattern, sni in EXIO_TO_SNI.items():
        if pattern in n:
            return sni
    return None

# ── 3. Aggregate EXIOBASE to SCB sector groups ───────────────────────────────
# F_ghg_SE  : (200,) kt CO2e, produktionsperspektiv
# cba_ghg_by_sec : (200,) kt CO2e, konsumtionsperspektiv
mapping = pd.DataFrame({
    "sni":     [map_sector(s) for s in F_ghg_SE.index],
    "exio_pba": F_ghg_SE.values / 1000,    # kt -> Mton
    "exio_cba": cba_ghg_by_sec.reindex(F_ghg_SE.index).fillna(0.0).values / 1000,
}, index=F_ghg_SE.index).dropna(subset=['sni'])

agg = mapping.groupby('sni')[['exio_pba', 'exio_cba']].sum()

# ── 4. Comparison table ───────────────────────────────────────────────────────
comp = scb_ref.join(agg, how='left')
comp['pba_diff_pct'] = (comp['exio_pba'] / comp['scb_pba'] - 1) * 100
comp['cba_diff_pct'] = (comp['exio_cba'] / comp['scb_cba'] - 1) * 100

def flag(pct):
    if pd.isna(pct):   return 'n/a '
    if abs(pct) <= 15: return 'OK  '
    if abs(pct) <= 30: return 'WARN'
    return                    'FAIL'

print('=' * 95)
print('CROSS-VALIDATION | Mton CO2e | Sweden 2022 | EXIOBASE 3.8.1 vs. SCB MI171 2025:5 Figur 1')
print('=' * 95)
print(f'  {"SNI":8s}  {"SCB pba":>8s}  {"EXIO pba":>9s}  {"diff%":>6s}  '
      f'{"SCB cba":>8s}  {"EXIO cba":>9s}  {"diff%":>6s}  Label')
print('-' * 95)
for sni, row in comp.iterrows():
    print(f'  {sni:8s}  {row["scb_pba"]:8.3f}  {row["exio_pba"]:9.3f}  '
          f'[{flag(row["pba_diff_pct"])}]{row["pba_diff_pct"]:+6.1f}%  '
          f'{row["scb_cba"]:8.3f}  {row["exio_cba"]:9.3f}  '
          f'[{flag(row["cba_diff_pct"])}]{row["cba_diff_pct"]:+6.1f}%  '
          f'{row["label"]}')
print('-' * 95)
pba_t_e = comp['exio_pba'].sum(); pba_t_s = comp['scb_pba'].sum()
cba_t_e = comp['exio_cba'].sum(); cba_t_s = comp['scb_cba'].sum()
print(f'  {"TOTAL":8s}  {pba_t_s:8.3f}  {pba_t_e:9.3f}  '
      f'[{flag((pba_t_e/pba_t_s-1)*100)}]{(pba_t_e/pba_t_s-1)*100:+6.1f}%  '
      f'{cba_t_s:8.3f}  {cba_t_e:9.3f}  '
      f'[{flag((cba_t_e/cba_t_s-1)*100)}]{(cba_t_e/cba_t_s-1)*100:+6.1f}%')
print('=' * 95)
print()
print('Interpretation notes:')
print('  Produktionsperspektiv (PBA) deviations: biogenic CO2 scope difference is the main')
print('  driver for C16 (wood) and C10-12 (food). SCB follows the Kyoto/national-inventory')
print('  boundary where biogenic CO2 is attributed to LULUCF, not to industry sectors.')
print('  Konsumtionsperspektiv (CBA) deviations: expected to be larger. SCB uses Swedish')
print('  national IO tables; EXIOBASE uses global IO. A consistent directional bias across')
print('  all sectors (all above or all below SCB) reflects a multiplier offset, not a data')
print('  error. Sector-specific outliers warrant investigation.')
print(f'\n  Sweden D_pba total (all 200 sectors): {F_ghg_SE.sum()/1000:.2f} Mton CO2e')
print(f'  Sweden D_cba total (all 200 sectors): {cba_ghg_by_sec.sum()/1000:.2f} Mton CO2e')
print('  Expected D_pba: ~45-50 Mton (Swedish national inventory 2022)')

## 12. Supply chain dependency analysis — direct foreign input linkages

This section computes direct first-tier foreign input linkages for Swedish sectors,
using the inter-industry transaction matrix **Z** directly (without the Leontief inverse).

**Purpose.** Unlike the three SCB perspectives (produktionsperspektiv, konsumtionsperspektiv,
and the value chain perspective to be added), this module answers a different set of questions:
which Swedish sectors act as *gateway sectors* through which foreign goods and services enter
the economy, and which foreign (country, sector) nodes are their direct first-tier suppliers?

**Two analytical views are computed.**

- **Section 12.1 (demand-attributed)** weights each Swedish sector's foreign purchases by its
  share of output serving Swedish final demand. This isolates the supply chain dependencies
  that are most directly relevant to Swedish consumption.

- **Section 12.2 (production-oriented)** covers all Swedish production regardless of where
  output goes. A downstream destination split shows what fraction of each gateway sector's
  output is retained in Sweden versus exported, indicating how reachable each sector is by
  Swedish or regional policy.

**Outputs feed NB03** (supply chain dependency and resilience visualisations) and the
Stockholm regional disaggregation, where the Z matrix is split into Stockholm and
Rest-of-Sweden flows to map regional supply chain exposure.

**Relationship to the full Leontief CBA.** Tier-1 totals are substantially smaller than
full-chain CBA/PBA totals because only the first supply-chain tier is captured. Both views
are valid; they answer different questions. The full CBA in sections 7-8 gives total
embedded impact; this section gives structural dependency at the direct procurement level.

In [ ]:
# Access or reconstruct the inter-industry transaction matrix Z.
t0 = time.time()
if getattr(exio, "Z", None) is not None:
    Z_df = normalize_frame(exio.Z, index_names=EXIO_RC_NAMES, column_names=EXIO_RC_NAMES)
    print("Z loaded directly from exio.Z")
else:
    x_flat = exio.x.squeeze()
    Z_raw  = exio.A.multiply(x_flat, axis=1)
    Z_df   = normalize_frame(Z_raw, index_names=EXIO_RC_NAMES, column_names=EXIO_RC_NAMES)
    print("Z reconstructed from A * diag(x)")
print(f"  Z_df shape: {Z_df.shape}  ({time.time()-t0:.1f}s)")

# Foreign-to-Sweden block: rows = foreign (country, sector); cols = 200 SE sectors
Z_imp_SE = (Z_df
    .drop(REGION, level="region", axis=0)
    .xs(REGION, level="region", axis=1))
print(f"  Z_imp_SE shape: {Z_imp_SE.shape}  (foreign sources x Swedish gateway sectors)")

# Physical intensity vectors for foreign sources (GHG and material only)
S_ghg_fgn = S_ghg.drop(REGION, level="region").reindex(Z_imp_SE.index).fillna(0.0)
S_mat_fgn = S_mat.sum(axis=0).drop(REGION, level="region").reindex(Z_imp_SE.index).fillna(0.0)
print(f"  Intensity vectors: {len(S_ghg_fgn)} foreign (country, sector) rows")


### 12.1 First-tier foreign inputs attributable to Swedish final demand

**What it measures.** For each Swedish sector whose output serves Swedish final demand, which foreign sectors does it purchase from directly? What direct expenditure, GHG, and material pressures are associated with those purchases?

**Allocation to Swedish final demand.** The CBA weight $w^{CBA}_k$ allocates each Swedish sector's foreign purchases in proportion to the share of that sector's output driven by Swedish final demand. This means only the portion of foreign inputs flowing through to domestic consumption is captured here.

**This is not a full CBA footprint.** It covers only direct (first-tier) foreign purchases. Deeper upstream tiers — for example, the energy and materials used to produce the foreign inputs — are excluded. The full Leontief-traced CBA in sections 7–8 includes all tiers.

**Method.** The tier-1 flow attributable to Swedish final demand from foreign source (i, s) through Swedish sector k is:

$$T1[i{\cdot}s,\ k] = Z\_imp\_SE[i{\cdot}s,\ k] \times w^{CBA}_k$$

where $w^{CBA}_k = x^{SE \to Swedish\_demand}_k \,/\, x^{SE\_total}_k$.

- **Economic dimension:** flow value in M EUR (no intensity weighting).
- **GHG and material:** multiplied by $S_d[i{\cdot}s]$ to convert to physical units.

Column sums give total tier-1 pressure channelled per Swedish gateway sector. Row sums give total tier-1 pressure from each foreign source.


In [ ]:
# CBA weight: fraction of each Swedish sector's output serving Swedish final demand
x_SE_cba = x_SE_total.xs(REGION, level="region")
x_SE_all  = exio.x.xs(REGION, level="region").squeeze()
x_SE_all  = x_SE_all.reindex(Z_imp_SE.columns).fillna(0.0)
x_SE_cba  = x_SE_cba.reindex(Z_imp_SE.columns).fillna(0.0)
cba_weight = (x_SE_cba / x_SE_all.replace(0, np.nan)).fillna(0.0).clip(0.0, 1.0)

# Tier-1 CBA matrices: (foreign_source x SE_gateway) per dimension
t0 = time.time()
# Economic: Z flows weighted by CBA share (no intensity multiplication)
tier1_cba_econ = Z_imp_SE.multiply(cba_weight, axis=1)
# Physical dimensions: Z flows * intensity at source * CBA share
tier1_cba_ghg  = Z_imp_SE.multiply(S_ghg_fgn, axis=0).multiply(cba_weight, axis=1)
tier1_cba_mat  = Z_imp_SE.multiply(S_mat_fgn, axis=0).multiply(cba_weight, axis=1)
print(f"Tier-1 (demand-attributed) matrices computed in {time.time()-t0:.1f}s.  Shape: {tier1_cba_ghg.shape}")

# Gateway summary: column sums
gateway_cba_econ = tier1_cba_econ.sum(axis=0).sort_values(ascending=False)
gateway_cba_ghg  = tier1_cba_ghg.sum(axis=0).sort_values(ascending=False)
gateway_cba_mat  = tier1_cba_mat.sum(axis=0).sort_values(ascending=False)

# Source summary: row sums
source_cba_econ = tier1_cba_econ.sum(axis=1).sort_values(ascending=False)
source_cba_ghg  = tier1_cba_ghg.sum(axis=1).sort_values(ascending=False)
source_cba_mat  = tier1_cba_mat.sum(axis=1).sort_values(ascending=False)

print(f"\n--- Tier-1 first-tier foreign inputs — demand-attributed totals ---")
print(f"  Foreign input expenditure (demand-attributed) : {gateway_cba_econ.sum():>10,.0f} M EUR")
print(f"  GHG                : {gateway_cba_ghg.sum():>10,.0f} kt CO2e")
print(f"  Material           : {gateway_cba_mat.sum():>10,.0f} kt")


In [ ]:
print(f"=== Top {TOP_N} Swedish gateway sectors — first-tier foreign inputs (demand-attributed), FOREIGN INPUT EXPENDITURE ===")
display(gateway_cba_econ.head(TOP_N).rename("Foreign input expenditure (M EUR)").to_frame())

print(f"\n=== Top {TOP_N} Swedish gateway sectors — first-tier foreign inputs (demand-attributed), GHG ===")
display(gateway_cba_ghg.head(TOP_N).rename("GHG (kt CO2e)").to_frame())

print(f"\n=== Top {TOP_N} Swedish gateway sectors — first-tier foreign inputs (demand-attributed), MATERIAL ===")
display(gateway_cba_mat.head(TOP_N).rename("Material (kt)").to_frame())

print(f"\n=== Top {TOP_N} foreign sources — first-tier foreign inputs (demand-attributed), FOREIGN INPUT EXPENDITURE ===")
display(pairs_table(source_cba_econ).rename(columns={"value": "Foreign input expenditure (M EUR)"}))

print(f"\n=== Top {TOP_N} foreign sources — first-tier foreign inputs (demand-attributed), GHG ===")
display(pairs_table(source_cba_ghg))

print(f"\n=== Top {TOP_N} foreign sources — first-tier foreign inputs (demand-attributed), MATERIAL ===")
display(pairs_table(source_cba_mat))


### 12.2 First-tier foreign input dependency of Swedish production

**What it measures.** For each Swedish sector — regardless of where its output goes — which foreign intermediate inputs does it purchase directly? What direct expenditure, GHG, and material pressures are associated with those purchases at the foreign source?

**This is not a PBA account.** In strict accounting terms, Swedish PBA covers only impacts physically occurring inside Sweden. The foreign upstream pressures computed here — at the source of Sweden's imported inputs — occur outside Swedish territory. This section is therefore better understood as a **production-oriented import linkage analysis**: it shows the first-tier foreign pressure associated with Swedish production activity, not Sweden's territorial footprint.

**Economic dimension:** Z flow values directly (no intensity weighting). Unit: M EUR.

**Downstream destination split.** For each Swedish sector, the fraction of its output that is exported versus retained in Sweden indicates the downstream reach of CE policy. Sectors with high domestic retention are more directly reachable by Swedish policy; export-oriented sectors primarily serve foreign demand chains.

$$\text{export\_share}[k] = \frac{\sum_j Z[SE_k, \text{foreign}_j] + \sum_{fd} Y[SE_k, \text{foreign}_{fd}]}{x^{SE\_total}_k}$$


In [ ]:
# Tier-1 PBA matrices: no CBA weighting
t0 = time.time()
# Economic: Z flows themselves (no intensity, no allocation)
tier1_pba_econ = Z_imp_SE.copy()
# Physical dimensions: Z flows * intensity at source
tier1_pba_ghg  = Z_imp_SE.multiply(S_ghg_fgn, axis=0)
tier1_pba_mat  = Z_imp_SE.multiply(S_mat_fgn, axis=0)
print(f"Tier-1 production-oriented import-linkage matrices computed in {time.time()-t0:.1f}s.  Shape: {tier1_pba_ghg.shape}")

gateway_pba_econ = tier1_pba_econ.sum(axis=0).sort_values(ascending=False)
gateway_pba_ghg  = tier1_pba_ghg.sum(axis=0).sort_values(ascending=False)
gateway_pba_mat  = tier1_pba_mat.sum(axis=0).sort_values(ascending=False)

source_pba_econ = tier1_pba_econ.sum(axis=1).sort_values(ascending=False)
source_pba_ghg  = tier1_pba_ghg.sum(axis=1).sort_values(ascending=False)
source_pba_mat  = tier1_pba_mat.sum(axis=1).sort_values(ascending=False)

print(f"\n--- Tier-1 production-oriented import-linkage totals ---")
print(f"  Foreign input expenditure : {gateway_pba_econ.sum():>10,.0f} M EUR")
print(f"  GHG                : {gateway_pba_ghg.sum():>10,.0f} kt CO2e")
print(f"  Material           : {gateway_pba_mat.sum():>10,.0f} kt")

# Downstream destination split
Z_SE_rows = Z_df.xs(REGION, level="region", axis=0)       # (200, 9800)
Y_SE_rows_pba = Y_df.xs(REGION, level="region", axis=0)   # (200, 343)

Z_to_fgn = Z_SE_rows.drop(REGION, level="region", axis=1)
Y_to_fgn = Y_SE_rows_pba.drop(REGION, level="region", axis=1)
Z_to_dom = Z_SE_rows.xs(REGION, level="region", axis=1)
Y_to_dom = Y_SE_rows_pba.xs(REGION, level="region", axis=1)

exported_use = Z_to_fgn.sum(axis=1) + Y_to_fgn.sum(axis=1)
domestic_use = Z_to_dom.sum(axis=1) + Y_to_dom.sum(axis=1)
total_use    = exported_use + domestic_use

export_share   = (exported_use / total_use.replace(0, np.nan)).fillna(0.0).clip(0.0, 1.0)
domestic_share = 1.0 - export_share
export_share   = export_share.reindex(Z_imp_SE.columns).fillna(0.0)
domestic_share = domestic_share.reindex(Z_imp_SE.columns).fillna(0.0)

_x_se_chk = exio.x.xs(REGION, level="region").squeeze().reindex(Z_imp_SE.columns).fillna(0.0)
_diff = ((total_use.reindex(Z_imp_SE.columns).fillna(0) - _x_se_chk) / _x_se_chk.replace(0, np.nan)).abs().max() * 100
print(f"\nDownstream split sanity: max |use_sum - x| / x = {_diff:.2f}%")

# Apply downstream split per dimension
gateway_pba_econ_exported = (tier1_pba_econ * export_share).sum(axis=0).sort_values(ascending=False)
gateway_pba_econ_domestic = (tier1_pba_econ * domestic_share).sum(axis=0).sort_values(ascending=False)
gateway_pba_ghg_exported  = (tier1_pba_ghg  * export_share).sum(axis=0).sort_values(ascending=False)
gateway_pba_ghg_domestic  = (tier1_pba_ghg  * domestic_share).sum(axis=0).sort_values(ascending=False)
gateway_pba_mat_exported  = (tier1_pba_mat  * export_share).sum(axis=0).sort_values(ascending=False)
gateway_pba_mat_domestic  = (tier1_pba_mat  * domestic_share).sum(axis=0).sort_values(ascending=False)

print(f"\n--- Downstream split (share of production-oriented import-linkage total) ---")
for label, exp_s, dom_s in [
    ("Foreign input exp.", gateway_pba_econ_exported, gateway_pba_econ_domestic),
    ("GHG",                gateway_pba_ghg_exported,  gateway_pba_ghg_domestic),
    ("Material",           gateway_pba_mat_exported,  gateway_pba_mat_domestic),
]:
    tot = exp_s.sum() + dom_s.sum()
    print(f"  {label:<22}: {exp_s.sum()/tot*100:5.1f}% exported, {dom_s.sum()/tot*100:5.1f}% domestic")


In [ ]:
print(f"=== Top {TOP_N} Swedish gateway sectors — production-oriented import linkage, FOREIGN INPUT EXPENDITURE ===")
gw_econ_split = pd.DataFrame({
    "Foreign input exp. (M EUR)": gateway_pba_econ,
    "Exported":      gateway_pba_econ_exported,
    "Domestic":      gateway_pba_econ_domestic,
}).nlargest(TOP_N, "Foreign input exp. (M EUR)")
display(gw_econ_split)

print(f"\n=== Top {TOP_N} Swedish gateway sectors — production-oriented import linkage, GHG ===")
gw_ghg_split = pd.DataFrame({
    "Total (kt CO2e)": gateway_pba_ghg,
    "Exported":        gateway_pba_ghg_exported,
    "Domestic":        gateway_pba_ghg_domestic,
}).nlargest(TOP_N, "Total (kt CO2e)")
display(gw_ghg_split)

print(f"\n=== Top {TOP_N} Swedish gateway sectors — production-oriented import linkage, MATERIAL ===")
gw_mat_split = pd.DataFrame({
    "Total (kt)": gateway_pba_mat,
    "Exported":   gateway_pba_mat_exported,
    "Domestic":   gateway_pba_mat_domestic,
}).nlargest(TOP_N, "Total (kt)")
display(gw_mat_split)

print(f"\n=== Top {TOP_N} foreign sources — production-oriented import linkage, FOREIGN INPUT EXPENDITURE ===")
display(pairs_table(source_pba_econ).rename(columns={"value": "Foreign input expenditure (M EUR)"}))

print(f"\n=== Top {TOP_N} foreign sources — production-oriented import linkage, GHG ===")
display(pairs_table(source_pba_ghg))

print(f"\n=== Top {TOP_N} foreign sources — production-oriented import linkage, MATERIAL ===")
display(pairs_table(source_pba_mat))


## 13. Export results


In [ ]:
print(f"Outputs written to {OUTPUT_DIR.resolve()}")
for p in sorted(OUTPUT_DIR.iterdir()):
    size_kb = p.stat().st_size / 1024
    print(f"  {p.name:<45}  {size_kb:>8,.1f} KB")


In [ ]:
import pickle

_to_save = {
    # ── Conventional PBA / CBA sector-level tables ────────────────────────────
    "pba_SE":             pba_SE,           # "Gross output (M EUR)", GHG, material
    "cba_SE":             cba_SE,           # "Direct expenditure (M EUR)", GHG, material
    "F_mat_SE":           F_mat_SE,
    "cba_mat_by_sec":     cba_mat_by_sec,
    # ── PBA economic gross output ─────────────────────────────────────────────
    "x_SE":               x_SE,             # (200,) Swedish gross output per sector
    # ── CBA source attribution ────────────────────────────────────────────────
    "src_econ_country":   src_econ_country, # (49,) direct expenditure by source country
    "src_ghg_country":    src_ghg_country,
    "src_mat_country":    src_mat_country,
    "src_econ":           src_econ,         # (9800,) direct expenditure by (country, sector)
    "src_ghg":            src_ghg,
    "src_mat_total":      src_mat_total,
    # ── PBA destination attribution ───────────────────────────────────────────
    "econ_dest_country":  econ_dest_country, # (49,) gross output by destination country
    "ghg_dest_country":   ghg_dest_country,
    "mat_dest_country":   mat_dest_country,
    "F_econ_dest":        F_econ_dest,       # (200, 49) direct sales per SE sector x country
    "F_ghg_dest":         F_ghg_dest,        # (200, 343) Leontief-traced GHG
    "F_mat_dest_total":   F_mat_dest_total,  # (200, 343) Leontief-traced material
    # ── Per-consumption-sector attribution matrices — Sankeys ─────────────────
    "attrib_econ":        attrib_econ,       # (9800, 200) direct expenditure
    "attrib_ghg":         attrib_ghg,        # (9800, 200) Leontief-traced GHG
    "attrib_mat_total":   attrib_mat_total,  # (9800, 200) Leontief-traced material
    # ── Swedish direct impact vectors ─────────────────────────────────────────
    "F_ghg_SE":           F_ghg_SE,
    "F_mat_total_SE":     F_mat_total_SE,
    # ── Supply chain dependency — demand-attributed (Section 12.1, feeds NB03) ─
    "tier1_cba_econ":     tier1_cba_econ,
    "tier1_cba_ghg":      tier1_cba_ghg,
    "tier1_cba_mat":      tier1_cba_mat,
    "gateway_cba_econ":   gateway_cba_econ,
    "gateway_cba_ghg":    gateway_cba_ghg,
    "gateway_cba_mat":    gateway_cba_mat,
    "source_cba_econ":    source_cba_econ,
    "source_cba_ghg":     source_cba_ghg,
    "source_cba_mat":     source_cba_mat,
    # ── Supply chain dependency — production-oriented (Section 12.2, feeds NB03) 
    "tier1_pba_econ":     tier1_pba_econ,
    "tier1_pba_ghg":      tier1_pba_ghg,
    "tier1_pba_mat":      tier1_pba_mat,
    "gateway_pba_econ":   gateway_pba_econ,
    "gateway_pba_ghg":    gateway_pba_ghg,
    "gateway_pba_mat":    gateway_pba_mat,
    "source_pba_econ":    source_pba_econ,
    "source_pba_ghg":     source_pba_ghg,
    "source_pba_mat":     source_pba_mat,
    # ── Production-oriented — downstream destination split ───────────────────
    "gateway_pba_econ_exported": gateway_pba_econ_exported,
    "gateway_pba_econ_domestic": gateway_pba_econ_domestic,
    "gateway_pba_ghg_exported":  gateway_pba_ghg_exported,
    "gateway_pba_ghg_domestic":  gateway_pba_ghg_domestic,
    "gateway_pba_mat_exported":  gateway_pba_mat_exported,
    "gateway_pba_mat_domestic":  gateway_pba_mat_domestic,
    "export_share":   export_share,
    "domestic_share": domestic_share,
    # ── Configuration ─────────────────────────────────────────────────────────
    "TOP_N": TOP_N,
    "TOP_C": TOP_C,
}

_pkl_path = OUTPUT_DIR / "analysis_objects.pkl"
with open(_pkl_path, "wb") as _f:
    pickle.dump(_to_save, _f, protocol=4)

print(f"Saved {len(_to_save)} objects -> {_pkl_path.resolve()}")
print("  NB02 objects: pba_SE, cba_SE, src_*, dest_*, attrib_*, F_*, x_SE")
print("  NB03 objects: tier1_*, gateway_*, source_*, export_share, domestic_share")
print(f"  File size: {_pkl_path.stat().st_size / 1024**2:.1f} MB")


---

## Summary — what this baseline produces for the Stockholm project

This notebook is the **national baseline** on which the Stockholm / Rest-of-Sweden disaggregation will be layered. For every re-run it writes a consistent set of tables to `./outputs/` and serialises all analysis objects to `analysis_objects.pkl` for use by the visualisation notebooks.

### A note on the economic dimension

The economic indicators in this notebook are **monetary scale measures**, not full upstream economic footprints. Gross output (production side) and direct expenditure (consumption side) are not Leontief-traced and are not directly comparable with the GHG and material CBA/PBA accounts. They indicate the monetary scale of production and consumption activity at each sector, which is a useful CE prioritisation lens, but overstating their equivalence to the physical footprint accounts should be avoided.

### CSV tables

- `pba_sweden_by_sector.csv` — 200 Swedish product sectors × 7 columns (gross output, total GHG, four Anthesis material categories, material total).
- `cba_sweden_by_sector.csv` — 200 Swedish consumption product categories × 7 columns (direct expenditure, total GHG, four Anthesis material categories, material total).
- `cba_source_by_country.csv` — 49 source countries × 3 indicators (direct expenditure, GHG, material).
- `pba_destination_by_country.csv` — 49 destination countries × 3 indicators (gross output, GHG, material).
- `sweden_pba_vs_cba.csv` — two-row net-trade table (GHG and material only).

### Analysis structure

**Sections 6–7** produce the production-side and consumption-side hotspot tables: top sectors per dimension.

**Section 8** attributes Sweden's consumption-side account to source countries and (country, sector) pairs. For the economic dimension, this is Sweden's direct expenditure — no Leontief tracing. For GHG and material, the full Leontief inverse traces impacts through all upstream tiers.

**Section 9** decomposes Sweden's production-side account by destination. For GHG and material, Leontief tracing attributes territorial impacts to final consumers. For the economic dimension, direct sales from Z and Y are used — these reach immediate buyers, not ultimate final consumers.

**Section 10** presents the GHG and material net-trade picture (PBA vs CBA). The economic dimension is excluded because gross output and direct expenditure measure different supply-chain stages and are not a valid net-trade pair.

**Section 12.1 (first-tier foreign inputs attributable to Swedish final demand)** identifies which Swedish sectors channel foreign inputs to serve Swedish consumption, using direct Z flows weighted by each sector's demand-side share of output. This is a first-tier diagnostic, not a full CBA.

**Section 12.2 (first-tier foreign input dependency of Swedish production)** identifies the direct foreign intermediate purchases of Swedish sectors regardless of where their output goes. This is a production-oriented import linkage analysis. The foreign upstream pressures computed here occur outside Swedish territory and are not Swedish PBA. A downstream split shows what fraction of each gateway sector's output is exported versus retained in Sweden.

**Section 11** cross-validates EXIOBASE GHG accounts against SCB MI171 2025:5 (Figur 1)
for the 14 Swedish manufacturing sectors reported across all three SCB perspectives.

**Section 13** exports all results to `./outputs/`.

### Headline numbers for Sweden, base year 2022

Headline PBA and CBA totals are printed by the code cells in sections 6–10. They are not repeated here as fixed numbers to avoid markdown becoming inconsistent if the notebook is rerun after any methodological change. The net-trade table in section 10 provides the GHG and material CBA/PBA ratios in computed form.

The economic dimension is not included in the net-trade comparison because gross output (production side) and direct expenditure (consumption side) measure different supply-chain stages and are not directly comparable as a net-trade pair.

### Known limitations of this baseline

1. **Biogenic versus fossil CO2 is not separated** in v3.8.2. The GHG results should be interpreted as EXIOBASE total GHG on the AR5 GWP100 basis, not as a fossil-only total.
2. **The Anthesis "recycled" material category has no direct equivalent** in EXIOBASE extractions.
3. **A small numerical discrepancy** (typically <0.5%) between Sweden's PBA GHG total and its destination-sum is a floating-point artefact with no substantive interpretation.
4. **The supply chain dependency analysis (Section 12) captures only the first supply-chain
   tier.** Totals are substantially smaller than full-chain CBA/PBA totals. Both views are
   correct; they answer different questions.
5. **The economic indicators are not full upstream economic footprints.** They indicate monetary scale; they do not trace where upstream economic activity embodied in products was generated.
